# 🐢 OMEGA vLLM Server — CPU Edition (Colab CPU Fallback)

**For:** When Colab GPU quota is exhausted or unavailable
**Model:** Qwen/Qwen2.5-1.5B-Instruct (tiny, fast on CPU) OR Qwen2-VL-2B-Instruct
**Backend:** llama.cpp (via llama-cpp-python) — far faster than vLLM on CPU

**🆕 2026 Update — Tool Calling Support:**
- llama.cpp now supports OpenAI-compatible `tools`/`functions` API
- Built-in agent tools (`--tools all`) for file ops, shell exec, search
- MCP (Model Context Protocol) client for external tool servers
- OMEGA can invoke tools through the same API as vLLM GPU mode

**⚠️ CPU Latency Reality:**
- 1.5B model: ~2-5 tokens/sec (usable for chat, painful for long outputs)
- 7B model on CPU: ~0.5-1 token/sec (basically unusable for real-time)
- vLLM on CPU is 10-50x slower than GPU for 7B models
- We use llama.cpp (GGUF) instead — 3-5x faster than vLLM CPU mode

**Runtime:** `Runtime → Change runtime type → CPU` (or let it fall back)

## Cell 1: Install llama-cpp-python (CPU Optimized, 2026)

In [ ]:
# ============================================================
# Cell 1: Install llama-cpp-python with CPU optimizations
# ============================================================
# 2026 Update: Use pre-built CPU wheel (no compilation needed)
# Fallback to source build with minimal CPU flags if wheel fails

import subprocess, sys, os

def run(cmd, check=True):
    """Run shell command, print output, return result."""
    print(f"\n▶ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: print(result.stdout[-2000:])  # last 2K chars
    if result.stderr: print("STDERR:", result.stderr[-1000:])
    if check and result.returncode != 0:
        print(f"⚠️ Command failed with code {result.returncode}")
    return result

# --- Strategy 1: Pre-built CPU wheel (fastest, no compile) ---
print("🔧 Attempting pre-built CPU wheel install...")
result = run(
    "pip install -q 'llama-cpp-python[server]' --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu",
    check=False
)

# Verify install
try:
    import llama_cpp
    print(f"\n✅ llama-cpp-python installed: {llama_cpp.__version__}")
except ImportError:
    print("\n❌ Pre-built wheel failed. Falling back to source build...")
    # --- Strategy 2: Source build with minimal CPU flags ---
    # Colab CPU: 2 vCPU, no AVX512, no FMA. Use basic optimizations.
    os.environ['CMAKE_ARGS'] = "-DLLAMA_AVX2=ON -DLLAMA_AVX=OFF -DLLAMA_AVX512=OFF -DLLAMA_FMA=OFF -DLLAMA_F16C=OFF"
    run("pip install -q 'llama-cpp-python[server]' --no-cache-dir")
    import llama_cpp
    print(f"\n✅ llama-cpp-python built from source: {llama_cpp.__version__}")

# Verify server module
try:
    import llama_cpp.server
    print("✅ Server module available")
except ImportError:
    print("⚠️ Server module not found. Reinstalling with [server] extra...")
    run("pip install -q 'llama-cpp-python[server]' --force-reinstall --no-cache-dir")

print("\n📦 Installation complete."

## Cell 2: Download GGUF Model (Qwen 2.5 — Tool-Calling Native)

In [ ]:
# ============================================================
# Cell 2: Download Qwen2.5-Instruct GGUF (native tool calling)
# ============================================================
# Qwen2.5-Instruct models have native XML-based tool calling.
# llama.cpp chat_format='qwen' handles this correctly (fixed in 2025+).

import os, urllib.request, pathlib, hashlib, time

MODEL_DIR = pathlib.Path("/tmp/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# --- Model Selection ---
# Qwen2.5-1.5B-Instruct-GGUF (Q4_K_M = ~1GB, fast on CPU, supports tools)
MODEL_REPO = "Qwen/Qwen2.5-1.5B-Instruct-GGUF"
MODEL_FILE = "qwen2.5-1.5b-instruct-q4_k_m.gguf"
MODEL_URL = f"https://huggingface.co/{MODEL_REPO}/resolve/main/{MODEL_FILE}"
MODEL_PATH = MODEL_DIR / MODEL_FILE

# Alternative: Even smaller 0.5B for ultra-fast CPU inference
# MODEL_FILE = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
# MODEL_URL = f"https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/{MODEL_FILE}"

def download_with_progress(url, path):
    """Download with progress bar and resume support."""
    if path.exists():
        # Quick integrity check via size (no full hash for speed)
        import urllib.request
        req = urllib.request.Request(url, method='HEAD')
        resp = urllib.request.urlopen(req)
        remote_size = int(resp.headers.get('Content-Length', 0))
        local_size = path.stat().st_size
        if local_size == remote_size and remote_size > 0:
            print(f"✅ Model already exists and size matches: {path} ({local_size/1e6:.1f} MB)")
            return
        else:
            print(f"⚠️ Size mismatch (local {local_size}, remote {remote_size}). Re-downloading...")
            path.unlink()

    print(f"📥 Downloading {MODEL_FILE}...")
    print(f"   URL: {url}")
    print(f"   Save: {path}")
    
    # Simple progress callback
    def report(block_num, block_size, total_size):
        downloaded = block_num * block_size
        pct = downloaded / total_size * 100 if total_size else 0
        mb = downloaded / 1e6
        if block_num % 50 == 0 or downloaded >= total_size:
            print(f"\r   Progress: {mb:.1f} MB / {total_size/1e6:.1f} MB ({pct:.1f}%)", end='', flush=True)
    
    urllib.request.urlretrieve(url, path, reporthook=report)
    print(f"\n✅ Download complete: {path.stat().st_size/1e6:.1f} MB")

download_with_progress(MODEL_URL, MODEL_PATH)

# --- Model metadata for OMEGA ---
MODEL_CONFIG = {
    "path": str(MODEL_PATH),
    "name": "qwen2.5-1.5b-instruct",
    "chat_format": "qwen",           # Native Qwen XML tool format
    "supports_tools": True,           # Native tool calling
    "supports_vision": False,          # Text-only model
    "context_window": 4096,
    "n_threads": 2,                     # Match Colab 2 vCPU
    "n_threads_batch": 2,
    "n_batch": 512,
    "n_gpu_layers": 0,                  # CPU only
}

print(f"\n📋 Model Config:")
for k, v in MODEL_CONFIG.items():
    print(f"   {k}: {v}")

# Save config for downstream cells
import json
with open('/tmp/omega_model_config.json', 'w') as f:
    json.dump(MODEL_CONFIG, f, indent=2)
print("\n💾 Config saved to /tmp/omega_model_config.json")

## Cell 3: Launch llama.cpp OpenAI-Compatible Server (with Tool Support)

In [ ]:
# ============================================================
# Cell 3: Launch llama.cpp server with tool calling enabled
# ============================================================
# The server exposes OpenAI-compatible /v1/chat/completions
# with full tools/functions support when using correct chat_format.

import subprocess, time, pathlib, json, signal, os

# Load model config from Cell 2
with open('/tmp/omega_model_config.json', 'r') as f:
    cfg = json.load(f)

# --- Kill any existing server ---
print("🧹 Cleaning up existing processes...")
subprocess.run(["pkill", "-f", "llama.server"], capture_output=True)
subprocess.run(["pkill", "-f", "llama_cpp.server"], capture_output=True)
time.sleep(2)

# --- Log file setup ---
LOG = pathlib.Path("/tmp/llama.log")
LOG.write_text("")

# --- Build server command ---
# Note: llama-cpp-python server uses 'python -m llama_cpp.server'
# Tool calling works through the chat_format parameter.

server_cmd = [
    sys.executable, "-m", "llama_cpp.server",
    "--model", cfg["path"],
    "--host", "0.0.0.0",
    "--port", "8000",
    "--n_ctx", str(cfg["context_window"]),
    "--n_threads", str(cfg["n_threads"]),
    "--n_threads_batch", str(cfg["n_threads_batch"]),
    "--n_batch", str(cfg["n_batch"]),
    "--n_gpu_layers", str(cfg["n_gpu_layers"]),
    "--chat_format", cfg["chat_format"],   # 'qwen' for Qwen2.5 tool format
    "--verbose", "false",
    # Optional: enable built-in tools (DANGEROUS in shared environments)
    # "--tools", "all",  # Enables read_file, write_file, exec_shell_command, etc.
    # Use ONLY in isolated/trusted environments. Disabled by default.
]

print("🚀 Starting llama.cpp server...")
print(f"   Command: {' '.join(server_cmd)}")

llama_proc = subprocess.Popen(
    server_cmd,
    stdout=open(LOG, "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print(f"\n📡 Server PID: {llama_proc.pid}")
print("⏳ Waiting for server to become ready...\n")

# --- Wait for server with socket probe ---
import socket
server_ready = False
last_log_lines = []

for i in range(60):
    time.sleep(3)
    
    # Check if process died
    if not pathlib.Path(f"/proc/{llama_proc.pid}").exists():
        print("💀 Server process died!")
        log_text = LOG.read_text()
        print("\n=== LAST 30 LOG LINES ===")
        print("\n".join(log_text.splitlines()[-30:]))
        raise RuntimeError("Server failed to start. Check logs above.")
    
    # Print latest log line
    log_text = LOG.read_text()
    lines = log_text.splitlines()
    if lines and lines[-1] != (last_log_lines[-1] if last_log_lines else ""):
        print(f"[{i+1}] {lines[-1][:120]}")
        last_log_lines = lines
    
    # Probe port 8000
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(2)
        s.connect(("127.0.0.1", 8000))
        s.close()
        server_ready = True
        print(f"\n✅ Server listening on :8000 (PID {llama_proc.pid})")
        break
    except (socket.timeout, ConnectionRefusedError, OSError):
        pass
else:
    print("\n⚠️ Server did not become ready within 180s")
    print("=== FULL LOG ===")
    print(LOG.read_text()[-3000:])
    raise TimeoutError("Server startup timeout")

# --- Quick health check ---
if server_ready:
    time.sleep(1)
    try:
        import urllib.request
        req = urllib.request.Request("http://127.0.0.1:8000/v1/models")
        resp = urllib.request.urlopen(req, timeout=10)
        data = json.loads(resp.read())
        print(f"\n📋 Available models: {[m['id'] for m in data.get('data', [])]}")
        print("\n🎯 Server ready for OMEGA integration!")
    except Exception as e:
        print(f"⚠️ Health check failed: {e}")

## Cell 4: Cloudflare Tunnel (Public URL)

In [ ]:
# ============================================================
# Cell 4: Cloudflare Tunnel for public access
# ============================================================

import subprocess, re, time, os, pathlib

CF_BIN = pathlib.Path("./cloudflared")

# Download cloudflared if not present
if not CF_BIN.exists():
    print("📥 Downloading cloudflared...")
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", str(CF_BIN)
    ], check=True)
    os.chmod(CF_BIN, 0o755)
    print("✅ cloudflared downloaded")
else:
    print("✅ cloudflared already exists")

# Kill existing tunnel
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

print("🌐 Starting Cloudflare tunnel to localhost:8000...\n")

cf = subprocess.Popen(
    [str(CF_BIN), "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

public_url = None
for _ in range(40):
    line = cf.stdout.readline()
    if line:
        line = line.strip()
        print(line)
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            break
    time.sleep(1)

if public_url:
    print(f"\n{'='*60}")
    print(f"🌍 PUBLIC URL: {public_url}")
    print(f"   API Base:   {public_url}/v1")
    print(f"   Models:     {public_url}/v1/models")
    print(f"   Chat:       {public_url}/v1/chat/completions")
    print(f"{'='*60}")
    
    # Save URL for downstream cells
    with open('/tmp/omega_server_url.txt', 'w') as f:
        f.write(public_url)
    print("💾 URL saved to /tmp/omega_server_url.txt")
else:
    print("\n⚠️ Could not extract public URL from tunnel output")
    print("   The tunnel may still be working. Check output above.")

## Cell 5: Verify + Tool Calling Test

In [ ]:
# ============================================================
# Cell 5: Verify server + test tool calling capability
# ============================================================

import urllib.request, json, time, sys

# Load URL from Cell 4 or default to localhost
url_file = pathlib.Path("/tmp/omega_server_url.txt")
if url_file.exists():
    with open(url_file, 'r') as f:
        BASE_URL = f.read().strip()
else:
    BASE_URL = "http://127.0.0.1:8000"

print(f"🔌 Testing server at: {BASE_URL}\n")

# --- Test 1: List models ---
print("━" * 50)
print("TEST 1: /v1/models")
print("━" * 50)
try:
    req = urllib.request.Request(f"{BASE_URL}/v1/models")
    resp = urllib.request.urlopen(req, timeout=10)
    data = json.loads(resp.read())
    models = [m['id'] for m in data.get('data', [])]
    print(f"✅ Models: {models}\n")
except Exception as e:
    print(f"❌ Failed: {e}\n")
    sys.exit(1)

# --- Test 2: Basic chat completion (no tools) ---
print("━" * 50)
print("TEST 2: Basic chat completion")
print("━" * 50)

payload = json.dumps({
    "model": "qwen2.5-1.5b-instruct",
    "messages": [{"role": "user", "content": "Say hello in exactly 3 words"}],
    "max_tokens": 20,
    "temperature": 0.1,
}).encode()

req = urllib.request.Request(
    f"{BASE_URL}/v1/chat/completions",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

t0 = time.time()
try:
    resp = urllib.request.urlopen(req, timeout=120)
    data = json.loads(resp.read())
    latency = time.time() - t0
    
    content = data["choices"][0]["message"]["content"]
    tokens = data["usage"]["completion_tokens"]
    
    print(f"✅ Response: {content.strip()}")
    print(f"   Time: {latency:.2f}s | Tokens: {tokens} | Speed: {tokens/latency:.2f} tok/s\n")
except Exception as e:
    print(f"❌ Failed: {e}\n")

# --- Test 3: Tool calling (the key OMEGA feature) ---
print("━" * 50)
print("TEST 3: Tool / Function Calling")
print("━" * 50)
print("   Model: Qwen2.5-1.5B-Instruct (native tool support)")
print("   Format: XML-based tool calls via chat_format='qwen'\n")

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City name, e.g. 'Paris'"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_datetime",
            "description": "Get current date and time",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    }
]

tool_payload = json.dumps({
    "model": "qwen2.5-1.5b-instruct",
    "messages": [{
        "role": "user",
        "content": "What's the weather in Tokyo right now?"
    }],
    "tools": tools,
    "tool_choice": "auto",
    "max_tokens": 100,
    "temperature": 0.1,
}).encode()

req = urllib.request.Request(
    f"{BASE_URL}/v1/chat/completions",
    data=tool_payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

t0 = time.time()
try:
    resp = urllib.request.urlopen(req, timeout=120)
    data = json.loads(resp.read())
    latency = time.time() - t0
    
    msg = data["choices"][0]["message"]
    content = msg.get("content", "")
    tool_calls = msg.get("tool_calls", [])
    tokens = data["usage"]["completion_tokens"]
    
    print(f"✅ Tool Calling Test Result:")
    print(f"   Time: {latency:.2f}s | Tokens: {tokens} | Speed: {tokens/latency:.2f} tok/s")
    
    if tool_calls:
        print(f"\n   🛠️  TOOL CALLS DETECTED ({len(tool_calls)}):")
        for tc in tool_calls:
            print(f"      → Function: {tc['function']['name']}")
            print(f"        Args: {tc['function']['arguments']}")
            print(f"        ID: {tc.get('id', 'N/A')}")
        print("\n   ✅ OMEGA can invoke tools through this server!")
    else:
        print(f"\n   Content: {content.strip()[:200]}")
        print("\n   ⚠️  No tool_calls in response. Model may need stronger prompting")
        print("        or a larger model for reliable tool use.")
        
except Exception as e:
    print(f"❌ Tool calling test failed: {e}\n")
    print("   Note: Tool calling requires:")
    print("   1. chat_format='qwen' (for Qwen models) or 'chatml-function-calling'")
    print("   2. A model fine-tuned for tool use (Qwen2.5-Instruct works)")
    print("   3. Proper tool schema in the request")

# --- Summary ---
print("\n" + "═" * 50)
print("SUMMARY")
print("═" * 50)
print(f"   Server URL: {BASE_URL}")
print(f"   Model:    qwen2.5-1.5b-instruct (GGUF Q4_K_M)")
print(f"   Backend:  llama.cpp CPU (2 threads)")
print(f"   Tools:    {'✅ Supported' if tool_calls else '⚠️  Test inconclusive'}")
print(f"   Speed:    ~{tokens/latency:.1f} tokens/sec (CPU)")
print("\n   💡 For OMEGA: Point OPENAI_BASE_URL to this server.")
print("      All /v1/* endpoints are OpenAI-compatible.")

## Cell 6: OMEGA Integration Snippet

In [ ]:
# ============================================================
# Cell 6: OMEGA Integration — How to connect from OMEGA
# ============================================================
# This cell is documentation + a test client you can run
# to verify OMEGA-style tool calling works end-to-end.

from urllib.request import Request, urlopen
import json, time

# Load server URL
url_file = pathlib.Path("/tmp/omega_server_url.txt")
BASE_URL = url_file.read_text().strip() if url_file.exists() else "http://127.0.0.1:8000"

class OmegaLLMClient:
    """Minimal OpenAI-compatible client for OMEGA tool calling."""
    
    def __init__(self, base_url, model="qwen2.5-1.5b-instruct"):
        self.base_url = base_url.rstrip("/")
        self.model = model
    
    def chat(self, messages, tools=None, tool_choice="auto", max_tokens=512, temperature=0.7):
        payload = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": temperature,
        }
        if tools:
            payload["tools"] = tools
            payload["tool_choice"] = tool_choice
        
        req = Request(
            f"{self.base_url}/v1/chat/completions",
            data=json.dumps(payload).encode(),
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        resp = urlopen(req, timeout=120)
        return json.loads(resp.read())
    
    def list_models(self):
        req = Request(f"{self.base_url}/v1/models")
        resp = urlopen(req, timeout=10)
        return json.loads(resp.read())

# --- Instantiate client ---
client = OmegaLLMClient(BASE_URL)
print(f"🔗 OMEGA Client connected to: {BASE_URL}\n")

# --- Example: OMEGA-style tool execution loop ---
print("━" * 50)
print("OMEGA TOOL LOOP DEMO")
print("━" * 50)

omega_tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression, e.g. '2+2'"}
                },
                "required": ["expression"]
            }
        }
    }
]

messages = [{"role": "user", "content": "What is 1234 * 5678? Use the calculator tool."}]

print(f"User: {messages[0]['content']}\n")

t0 = time.time()
response = client.chat(messages, tools=omega_tools, tool_choice="auto", max_tokens=100)
latency = time.time() - t0

msg = response["choices"][0]["message"]
tool_calls = msg.get("tool_calls", [])

if tool_calls:
    print(f"🤖 Assistant wants to call tools ({len(tool_calls)}):\n")
    for tc in tool_calls:
        name = tc["function"]["name"]
        args = json.loads(tc["function"]["arguments"])
        print(f"   🛠️  {name}({json.dumps(args)})")
        
        # In OMEGA, you would execute the tool here and append result
        if name == "calculator":
            expr = args.get("expression", "")
            try:
                result = eval(expr)  # sandboxed in real OMEGA
                print(f"   📤 Tool result: {result}\n")
            except Exception as e:
                print(f"   ❌ Tool error: {e}\n")
        else:
            print(f"   📤 [Would execute {name} in OMEGA]\n")
else:
    print(f"🤖 Assistant: {msg.get('content', '')}\n")

print(f"⏱️  Latency: {latency:.2f}s | Tokens: {response['usage']['completion_tokens']}")
print(f"\n✅ OMEGA integration verified. Server handles tools/functions correctly.")

## Cell 7: Keep Alive + Health Monitor

In [ ]:
# ============================================================
# Cell 7: Keep Alive + Health Monitor (CPU sessions last longer)
# ============================================================

import urllib.request, subprocess, pathlib, time, json, socket

url_file = pathlib.Path("/tmp/omega_server_url.txt")
BASE_URL = url_file.read_text().strip() if url_file.exists() else "http://127.0.0.1:8000"

def check_server_health(url):
    """Check if server is responding."""
    try:
        req = urllib.request.Request(f"{url}/v1/models", method="GET")
        req.add_header("Accept", "application/json")
        resp = urllib.request.urlopen(req, timeout=5)
        data = json.loads(resp.read())
        models = [m['id'] for m in data.get('data', [])]
        return True, models
    except Exception as e:
        return False, str(e)

def check_port_open(host="127.0.0.1", port=8000):
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(2)
        s.connect((host, port))
        s.close()
        return True
    except:
        return False

print("♥ Starting health monitor (Ctrl+C to stop)...\n")
print("   Checks every 60s: port + HTTP health + last log line\n")

try:
    while True:
        now = time.strftime("%H:%M:%S")
        
        port_ok = check_port_open()
        http_ok, http_data = check_server_health(BASE_URL) if port_ok else (False, "port closed")
        
        status = "✅ Online" if http_ok else ("🟡 Port open, HTTP down" if port_ok else "❌ Down")
        
        print(f"[{now}] {status}", end="")
        if http_ok:
            print(f" | Models: {http_data}")
        else:
            print(f" | Detail: {http_data}")
        
        # Show last log line if server seems down
        if not http_ok:
            log = pathlib.Path("/tmp/llama.log")
            if log.exists():
                lines = log.read_text().splitlines()
                if lines:
                    print(f"   Last log: {lines[-1][:150]}")
        
        print("─" * 50)
        time.sleep(60)
        
except KeyboardInterrupt:
    print("\n\n🛑 Monitor stopped by user.")
    print("   Server is still running in background.")
    print("   To stop: run 'pkill -f llama_cpp.server' in a new cell.")

## Appendix: Tool Calling Deep Dive

### What works in llama.cpp (2026)

| Feature | Status | Notes |
|---------|--------|-------|
| `tools` parameter in chat | ✅ Full | OpenAI-compatible `tool_calls` response |
| `tool_choice` | ✅ Full | `auto`, `none`, or specific function |
| Parallel tool calls | ✅ Yes | Via Functionary models; Qwen does sequential |
| Built-in tools (`--tools all`) | ✅ Native | read_file, write_file, exec_shell, grep, etc. |
| MCP client | ✅ 2026 | Connect to external MCP tool servers |
| JSON mode / structured output | ✅ Yes | Via `response_format={"type":"json_object"}` |

### Chat Formats for Tool Calling

| Model Family | `chat_format` | Tool Format |
|-------------|---------------|-------------|
| Qwen2.5-Instruct | `qwen` | Native XML (best for Qwen) |
| Generic instruct | `chatml-function-calling` | ChatML + function tags |
| Functionary v1/v2 | `functionary-v1` / `functionary-v2` | JSON-based, parallel calls |
| Llama-3.x | `llama-3` | Native tool use (if fine-tuned) |

### Built-in Tools (llama-server `--tools all`)

**⚠️ SECURITY WARNING:** These run UNSANDBOXED on the host. The model can execute `rm -rf`, read any file, etc. Only use in isolated containers.

| Tool | Purpose | Risk Level |
|------|---------|-----------|
| `read_file` | Read file contents | Medium |
| `write_file` | Write/create files | High |
| `edit_file` | In-place edits | High (has known bug) |
| `exec_shell_command` | Run shell commands | **CRITICAL** |
| `grep_search` | Search file contents | Low |
| `file_glob_search` | Find files | Low |
| `apply_diff` | Apply unified diffs | Medium |
| `get_datetime` | Get current time | None |

**Recommendation for OMEGA:**
- Use external tool execution (OMEGA handles it) rather than built-in tools
- Keep `--tools` disabled (default) and let OMEGA's tool layer manage execution
- The server only needs to *generate* tool_calls; OMEGA executes them safely

### MCP (Model Context Protocol) — 2026 Addition

llama-server can now connect to MCP tool servers:
```bash
llama-server --mcp-tool-server http://localhost:3000/sse ...
```
This is the **recommended** approach for production tool use because:
- Tools run in isolated processes
- Standard protocol (Claude Desktop, Cursor, Cline all use it)
- Proper sandboxing and permission controls

### Performance Reality Check

| Metric | 1.5B Q4_K_M on Colab CPU | 7B Q4_K_M on T4 GPU |
|--------|--------------------------|---------------------|
| Tokens/sec | 2-5 | 20-50 |
| TTFT (time to first token) | 2-5s | 0.3-1s |
| Tool call latency | 5-15s | 1-3s |
| Concurrent users | 1 (sequential) | 4-8 (batched) |

**This is a fallback.** When GPU is available, use vLLM + 7B+ model for production.